# WDC Schema.org Class Analysis — ClaimReview, Event & NewsArticle

**Source:** [Web Data Commons 2024-12 — Schema.org Class-Specific Subsets](https://webdatacommons.org/structureddata/2024-12/stats/schema_org_subsets.html)

## Datasets

| Class         | Source                                    | Size (compressed) | Part files |
|---------------|-------------------------------------------|-------------------|------------|
| `ClaimReview` | Dedicated WDC subset                      | ~56 MB            | 1          |
| `Event`       | Dedicated WDC subset                      | ~20.83 GB         | 133        |
| `NewsArticle` | *Extracted from* the RadioStation subset  | ~161 MB           | 1          |

### Why NewsArticle is different

WDC does **not** publish a dedicated `NewsArticle` subset. However, the
RadioStation subset lists NewsArticle as one of its top related classes
(201,586 quads) — because radio-station websites publish news articles on
the same pages as their station metadata. We download the RadioStation
subset and filter to NewsArticle entities inside it.

**Important caveat:** this is a *biased sample* of NewsArticle. It contains
only articles that appear on pages which also host RadioStation markup, i.e.
articles published by radio stations. Real-world NewsArticle data is
much broader (NYT, BBC, blogs, etc.) and would require scraping NewsArticle
instances out of other subsets or re-extracting from Common Crawl.

All three subsets contain **merged JSON-LD + Microdata** — the class-specific
dump does not tag each quad with the extractor that produced it, so we cannot
separate the two formats here.

## Analyses (run in the same order for all three classes, results shown side-by-side)

1. **Prevalence & Domain Distribution** — how many entities, where do they live
2. **Namespace Correctness** — canonical `schema.org` URIs vs. malformed / other
3. **Type Validity** — does the declared class exist in schema.org?
4. **Property Validity** — is the property actually allowed on this type?
5. **Domain / Range Violations** — does each value fit the property's expected type?
6. **Property Coverage** — which fraction of the vocabulary is actually used?
7. **AvP (Average Properties per Entity)** — annotation richness
8. **Characteristic Sets** — the most common property combinations in the wild

For each analysis, all three classes print consecutively. A consolidated
summary table at the bottom collects every metric across all three.

## Part 0 — Shared Setup

Imports, constants, and cache paths used by every downstream cell.

In [ ]:
# !pip install requests pandas tqdm

In [1]:
# ── Standard library ─────────────────────────────────────────────────────────
import requests, gzip, re, json, time, statistics
from collections import defaultdict, Counter
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

# ── WDC endpoints ────────────────────────────────────────────────────────────
# The class-specific subset URL pattern is:
#   {BASE}/{ClassName}/part_{N}.gz
# where part_0 is always present and part_N continues until a 404.
WDC_CLASS_BASE = "https://data.dws.informatik.uni-mannheim.de/structureddata/2024-12/quads/classspecific"

# ── Schema.org namespaces ────────────────────────────────────────────────────
# Both HTTP and HTTPS forms are canonical — publishers use either inconsistently,
# so we accept both and treat anything else (trailing slashes, wrong case,
# typos like "schema.org/" with extra segments) as malformed.
SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

# ── Local cache layout ───────────────────────────────────────────────────────
# Everything is cached under ./wdc_class_data/ so re-running the notebook
# does NOT re-download the ~21 GB Event corpus.
DATA_DIR = Path("wdc_class_data")
DATA_DIR.mkdir(exist_ok=True)

CLAIMREVIEW_NQ   = DATA_DIR / "ClaimReview.nq.gz"     # single part
EVENT_PARTS_DIR  = DATA_DIR / "Event"                 # directory of 133 parts
EVENT_PARTS_DIR.mkdir(exist_ok=True)

# RadioStation is our source for NewsArticle — NewsArticle has no dedicated
# WDC subset, but appears in the RadioStation dump at ~200K quads because
# radio-station pages often embed news articles.
RADIOSTATION_NQ  = DATA_DIR / "RadioStation.nq.gz"    # single part

# ── Global summary collector ─────────────────────────────────────────────────
# Every analysis function writes one or more metrics here; the final section
# of the notebook prints them as a single side-by-side comparison table.
# Structure: SUMMARY[class_name][metric_name] = value_or_string
SUMMARY: dict[str, dict[str, object]] = {
    "ClaimReview": {},
    "Event":       {},
    "NewsArticle": {},
}

print("Setup complete.")
print(f"  ClaimReview   → {CLAIMREVIEW_NQ}")
print(f"  Event parts   → {EVENT_PARTS_DIR}/")
print(f"  RadioStation  → {RADIOSTATION_NQ}   (source for NewsArticle)")

Setup complete.
  ClaimReview   → wdc_class_data/ClaimReview.nq.gz
  Event parts   → wdc_class_data/Event/
  RadioStation  → wdc_class_data/RadioStation.nq.gz   (source for NewsArticle)


## Part 1 — Download Class-Specific Subsets

One function covers both cases:
- **Single part** (ClaimReview, RadioStation): just grab `part_0.gz`.
- **Multi-part** (Event): keep fetching `part_N.gz` until the server returns 404.

Both skip files that are already cached, so this cell is safe to re-run.

We download **RadioStation** rather than NewsArticle directly, because
NewsArticle has no dedicated WDC subset. NewsArticle entities will be filtered
out of the RadioStation dump in Part 4.

In [2]:
def download_part(class_name: str, part: int, dest: Path) -> bool:
    """
    Download one WDC class-specific part file.
    Returns True on success/cache-hit, False if the part does not exist (404/403).
    """
    # Short-circuit if we already have it — lets us resume interrupted runs.
    if dest.exists():
        print(f"  [part {part}] cached ({dest.stat().st_size / 1e6:.1f} MB)")
        return True

    url = f"{WDC_CLASS_BASE}/{class_name}/part_{part}.gz"

    # HEAD first so multi-part loops know when to stop without wasting bandwidth.
    head = requests.head(url, timeout=30)
    if head.status_code in (403, 404):
        return False
    head.raise_for_status()

    print(f"  [part {part}] downloading {url} …")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        done  = 0
        # Stream in 1 MB chunks so even the 21 GB Event download stays bounded
        # in memory; we never hold the whole part in RAM.
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r    {done/1e6:.1f} / {total/1e6:.1f} MB", end="", flush=True)
    print(f"\n  [part {part}] saved → {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return True


def download_class_subset(class_name: str, dest: Path | None = None,
                          dest_dir: Path | None = None) -> list[Path]:
    """
    Unified downloader for single-part and multi-part class subsets.
      - `dest`     : if given, download only part_0 to this path (single-part mode).
      - `dest_dir` : if given, download every part_N.gz into this directory.
    Returns the list of local part paths.
    """
    if dest is not None:
        download_part(class_name, 0, dest)
        return [dest]

    assert dest_dir is not None, "must pass dest or dest_dir"
    paths, part = [], 0
    while True:
        p = dest_dir / f"part_{part}.gz"
        if not download_part(class_name, part, p):
            print(f"  [part {part}] 404/403 — stopping. {part} parts total.")
            break
        paths.append(p)
        part += 1
    return paths


# ── ClaimReview: one small file ──────────────────────────────────────────────
print("ClaimReview:")
cr_parts = download_class_subset("ClaimReview", dest=CLAIMREVIEW_NQ)

# ── Event: many parts (this is the heavy download — only runs if not cached) ─
print("\nEvent:")
event_parts = download_class_subset("Event", dest_dir=EVENT_PARTS_DIR)
total_size = sum(p.stat().st_size for p in event_parts) / 1e9
print(f"  Event total: {len(event_parts)} parts, {total_size:.2f} GB on disk")

# ── RadioStation: one small file, our NewsArticle source ─────────────────────
print("\nRadioStation (→ NewsArticle source):")
rs_parts = download_class_subset("RadioStation", dest=RADIOSTATION_NQ)

ClaimReview:
  [part 0] cached (57.6 MB)

Event:
  [part 0] cached (162.2 MB)
  [part 1] cached (159.9 MB)
  [part 2] cached (181.1 MB)
  [part 3] cached (132.5 MB)
  [part 4] cached (156.3 MB)
  [part 5] cached (134.0 MB)
  [part 6] cached (123.6 MB)
  [part 7] cached (217.4 MB)
  [part 8] cached (150.5 MB)
  [part 9] cached (140.9 MB)
  [part 10] cached (419.6 MB)
  [part 11] cached (146.8 MB)
  [part 12] cached (124.2 MB)
  [part 13] cached (141.7 MB)
  [part 14] cached (142.0 MB)
  [part 15] cached (128.8 MB)
  [part 16] cached (142.4 MB)
  [part 17] cached (128.1 MB)
  [part 18] cached (127.7 MB)
  [part 19] cached (163.7 MB)
  [part 20] cached (149.5 MB)
  [part 21] cached (129.6 MB)
  [part 22] cached (115.1 MB)
  [part 23] cached (160.9 MB)
  [part 24] cached (145.3 MB)
  [part 25] cached (155.0 MB)
  [part 26] cached (936.4 MB)
  [part 27] cached (186.5 MB)
  [part 28] cached (130.2 MB)
  [part 29] cached (127.2 MB)
  [part 30] cached (131.8 MB)
  [part 31] cached (141.9 MB)
 

## Part 2 — Parse N-Quads into an Entity Store

We stream the `.nq.gz` files line-by-line and collapse every quad into a
per-subject record:

```
entity_store[subject] = {
    "types":  {class_URI, …},                            # all rdf:type assertions
    "props":  {pred_URI: {obj_str: {source_page_URL…}}}, # non-type props + provenance
    "graphs": {source_page_URL, …},                      # every named graph the entity appears in
}
```

The inner mapping on `props` keeps triple-level provenance: each `(predicate, object)`
remembers *which source page* asserted it. This is essential for deduplicating
entities that appear on near-duplicate URLs (e.g. canonical vs. query-string variants).

In [ ]:
import os
import multiprocessing as _mp

# ── Optional: Intel ISA-L fast gzip (pip install isal) ───────────────────────
try:
    from isal import igzip as _igzip
    _ISAL = True
    print("isal available — fast gzip enabled")
except ImportError:
    _ISAL = False
    print("isal not found — using stdlib gzip  (pip install isal for ~2-3× speedup)")

# ── N-Quads regex (kept for reference; hot path now uses the string parser) ──
NQ_RE = re.compile(
    r'^'
    r'(<[^>]*>|_:\S+)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+'
    r'(<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@\w[\w-]*|(?:\^\^<[^>]+>))?)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+\.\s*$'
)


def parse_nq_line(line: str):
    """
    Fast N-Quads parser using string operations.

    Terminates with '.' possibly preceded by whitespace (N-Quads allows both
    '<G> .' and '<G>.').  Graph is always the last URI token, so rfind(' <')
    reliably locates it even when literals contain '<' characters.
    Returns (subj, pred, obj, graph) or None.
    """
    ln = line.rstrip()
    if not ln.endswith('.'):
        return None
    ln = ln[:-1].rstrip()        # drop '.' then any whitespace before it

    if not ln.endswith('>'):
        return None              # graph must be a URI

    # Graph: last '<…>' token, preceded by a space
    gs = ln.rfind(' <')
    if gs < 0:
        return None
    graph = ln[gs + 1:]
    ln = ln[:gs].rstrip()

    # Subject (first token — no spaces inside URIs / blank nodes)
    i = ln.find(' ')
    if i < 0:
        return None
    subj = ln[:i]
    rest = ln[i + 1:]

    # Predicate (second token)
    j = rest.find(' ')
    if j < 0:
        return None
    pred = rest[:j]
    obj  = rest[j + 1:]

    return (subj, pred, obj, graph) if obj else None


def strip_angle(s: str) -> str:
    """<http://foo> → http://foo  (no-op for blank nodes / literals)."""
    return s[1:-1] if s.startswith("<") and s.endswith(">") else s


def _make_entity():
    return {
        "types":  set(),
        "props":  defaultdict(lambda: defaultdict(set)),
        "graphs": set(),
    }


# ── Per-file worker ───────────────────────────────────────────────────────────
def _parse_one_file(task):
    """Parse one .nq.gz file; return (plain_store, errors, lines, bad_lines)."""
    path, capture_errors = task
    _RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

    try:
        from isal import igzip
        open_fn = lambda p: igzip.open(str(p), "rt", encoding="utf-8", errors="replace")
    except ImportError:
        import gzip as _gz
        open_fn = lambda p: _gz.open(p, "rt", encoding="utf-8", errors="replace")

    # Inline fast parser — same logic as parse_nq_line above
    def _parse(line):
        ln = line.rstrip()
        if not ln.endswith('.'):
            return None
        ln = ln[:-1].rstrip()
        if not ln.endswith('>'):
            return None
        gs = ln.rfind(' <')
        if gs < 0:
            return None
        graph = ln[gs + 1:]
        ln = ln[:gs].rstrip()
        i = ln.find(' ')
        if i < 0:
            return None
        subj = ln[:i]
        rest = ln[i + 1:]
        j = rest.find(' ')
        if j < 0:
            return None
        pred, obj = rest[:j], rest[j + 1:]
        return (subj, pred, obj, graph) if obj else None

    store  = {}
    errors = lines = 0
    bad    = []

    with open_fn(path) as f:
        for line in f:
            lines += 1
            p = _parse(line)
            if not p:
                errors += 1
                if capture_errors and len(bad) < 50:
                    bad.append(line.strip())
                continue
            subj, pred, obj, graph = p
            graph_uri = graph[1:-1]
            pred_uri  = pred[1:-1]

            if subj not in store:
                store[subj] = {"types": set(), "props": {}, "graphs": set()}
            es = store[subj]
            es["graphs"].add(graph_uri)
            if pred_uri == _RDF_TYPE:
                es["types"].add(obj[1:-1] if obj.startswith('<') else obj)
            else:
                props = es["props"]
                if pred_uri not in props:
                    props[pred_uri] = {}
                op = props[pred_uri]
                if obj not in op:
                    op[obj] = set()
                op[obj].add(graph_uri)

    return store, errors, lines, bad


def _merge(base: dict, partial: dict):
    """Merge partial store into base in-place."""
    for subj, data in partial.items():
        if subj not in base:
            base[subj] = data
        else:
            es = base[subj]
            es["types"]  |= data["types"]
            es["graphs"] |= data["graphs"]
            for pred, omap in data["props"].items():
                if pred not in es["props"]:
                    es["props"][pred] = omap
                else:
                    bp = es["props"][pred]
                    for obj, gs in omap.items():
                        if obj not in bp:
                            bp[obj] = gs
                        else:
                            bp[obj] |= gs


def build_entity_store(paths, capture_errors: bool = False, n_workers: int | None = None):
    """
    Unified builder — single Path or list of Paths.
    n_workers: None = auto (min(cpu_count, 8, n_files)), 1 = serial.
    """
    if isinstance(paths, Path):
        paths = [paths]

    n      = n_workers if n_workers is not None else min(os.cpu_count() or 1, len(paths), 8)
    use_mp = (n > 1 and len(paths) > 1)

    store        = {}
    total_errors = total_lines = 0
    all_bad: list = []
    tasks = [(p, capture_errors) for p in paths]

    t0 = time.time()

    if use_mp:
        ctx  = _mp.get_context("fork")
        done = 0
        with ctx.Pool(processes=n) as pool:
            for partial, errors, lines, bad in pool.imap_unordered(_parse_one_file, tasks):
                _merge(store, partial)
                total_errors += errors
                total_lines  += lines
                all_bad.extend(bad[:max(0, 50 - len(all_bad))])
                done += 1
                print(f"  merged {done}/{len(paths)}  —  "
                      f"{total_lines:,} lines, {len(store):,} entities", flush=True)
    else:
        for path, task in zip(paths, tasks):
            partial, errors, lines, bad = _parse_one_file(task)
            _merge(store, partial)
            total_errors += errors
            total_lines  += lines
            all_bad.extend(bad[:max(0, 50 - len(all_bad))])
            if len(paths) > 1:
                print(f"  done: {path.name}  (running: {total_lines:,} lines, {len(store):,} entities)")

    elapsed = time.time() - t0
    rate    = total_lines / elapsed if elapsed else 0
    print(f"  Parse time   : {elapsed:.1f}s  ({rate:,.0f} lines/s)")
    return store, total_errors, total_lines, all_bad


# ── Build ClaimReview store ───────────────────────────────────────────────────
print("Building ClaimReview entity store …")
cr_store, cr_errors, cr_total, cr_bad = build_entity_store(CLAIMREVIEW_NQ, capture_errors=True)
print(f"  Lines parsed : {cr_total:,}")
print(f"  Parse errors : {cr_errors:,}")
print(f"  Entities     : {len(cr_store):,}")

# ── Build Event store (133 files, parallelised) ───────────────────────────────
print("\nBuilding Event entity store …")
ev_store, ev_errors, ev_total, ev_bad = build_entity_store(event_parts, capture_errors=False)
print(f"\n  Lines parsed : {ev_total:,}")
print(f"  Parse errors : {ev_errors:,}")
print(f"  Entities     : {len(ev_store):,}")

# ── Build RadioStation store ──────────────────────────────────────────────────
print("\nBuilding RadioStation entity store (for NewsArticle extraction) …")
rs_store, rs_errors, rs_total, rs_bad = build_entity_store(RADIOSTATION_NQ, capture_errors=True)
print(f"  Lines parsed : {rs_total:,}")
print(f"  Parse errors : {rs_errors:,}")
print(f"  Entities     : {len(rs_store):,}")

isal available — fast gzip enabled
Building ClaimReview entity store …
  Parse time   : 25.1s  (156,289 lines/s)
  Lines parsed : 3,919,703
  Parse errors : 0
  Entities     : 787,354

Building Event entity store …


### Sample parse errors

The parser accepts only strict N-Quads. Things that break it in practice:
malformed URIs with embedded tags (e.g. `<https://schema.org/url</div>>`),
unescaped quotes inside literals, missing trailing `.`. A handful is expected
at this scale; anything more than ~0.1% of lines is worth investigating.

In [ ]:
if cr_bad:
    print("ClaimReview sample bad lines:")
    for line in cr_bad[:3]:
        print(f"  {line!r}")
else:
    print("ClaimReview: no parse errors captured.")

## Part 3 — Load Schema.org Vocabulary

Fetch the full schema.org vocabulary (JSON-LD) once, cache it locally, and
build four lookup structures:

| Structure         | Key                | Value                                     |
|-------------------|--------------------|-------------------------------------------|
| `so_types`        | —                  | set of *all* class names                  |
| `so_type_props`   | class name         | set of valid properties (incl. inherited) |
| `so_prop_range`   | property name      | set of expected value types               |
| `so_type_parents` | class name         | set of direct superclasses                |

Property inheritance is resolved by walking `rdfs:subClassOf` edges transitively,
so `so_type_props["ClaimReview"]` correctly contains properties inherited from
`Review` → `CreativeWork` → `Thing`.

In [ ]:
VOCAB_CACHE = DATA_DIR / "schemaorg_vocab.json"

def load_schemaorg_vocab():
    """
    Fetch (or load from cache) the schema.org JSON-LD dump and derive:
      - all_types       : set of every class name
      - type_props      : {class: {prop, ...}} WITH inherited properties
      - prop_range      : {prop: {expected_type, ...}}
      - type_parents    : {class: {parent_class, ...}}  (direct only)
    """
    if VOCAB_CACHE.exists():
        print("Loading vocab from cache …")
        with open(VOCAB_CACHE) as f:
            raw = json.load(f)
    else:
        url = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
        print(f"Fetching schema.org vocab from {url} …")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.json()
        with open(VOCAB_CACHE, "w") as f:
            json.dump(raw, f)
        print(f"  Cached → {VOCAB_CACHE}")

    graph = raw.get("@graph", [])
    print(f"  {len(graph):,} vocabulary items")

    # Helper: pull the local name out of a URI or compact form.
    # Handles both https://schema.org/ClaimReview and schema:ClaimReview.
    def local(uri: str) -> str:
        return uri.split("/")[-1].split("#")[-1].split(":")[-1]

    # Helper: domainIncludes/rangeIncludes can be a dict, list of dicts, or string.
    # Normalise to a list of @id strings.
    def ids_of(val) -> list[str]:
        if isinstance(val, dict):
            return [val.get("@id", "")]
        if isinstance(val, list):
            return [v.get("@id", "") if isinstance(v, dict) else str(v) for v in val]
        return [str(val)]

    all_types    = set()
    type_props   = defaultdict(set)
    prop_range   = defaultdict(set)
    type_parents = defaultdict(set)

    # ── Pass 1: classes (both rdfs:Class and schema:DataType count) ──────────
    for item in graph:
        item_type = str(item.get("@type", ""))
        if "rdfs:Class" in item_type or "schema:DataType" in item_type:
            name = local(item.get("@id", ""))
            all_types.add(name)
            for parent_id in ids_of(item.get("rdfs:subClassOf", [])):
                if parent_id:
                    type_parents[name].add(local(parent_id))

    # ── Pass 2: properties and their domains/ranges ──────────────────────────
    for item in graph:
        if "rdf:Property" in str(item.get("@type", "")):
            prop_name = local(item.get("@id", ""))
            for d in ids_of(item.get("schema:domainIncludes", [])):
                if d:
                    type_props[local(d)].add(prop_name)
            for r in ids_of(item.get("schema:rangeIncludes", [])):
                if r:
                    prop_range[prop_name].add(local(r))

    # ── Pass 3: propagate properties down the class hierarchy ────────────────
    # A property declared on CreativeWork is implicitly valid on ClaimReview.
    # Memoised DFS so we only compute each ancestor set once.
    def get_ancestors(t, memo={}):
        if t in memo:
            return memo[t]
        ancestors = set()
        for parent in type_parents.get(t, set()):
            ancestors.add(parent)
            ancestors |= get_ancestors(parent, memo)
        memo[t] = ancestors
        return ancestors

    for t in list(all_types):
        for ancestor in get_ancestors(t):
            type_props[t] |= type_props.get(ancestor, set())

    print(f"  Classes : {len(all_types):,}")
    print(f"  With properties defined : {sum(1 for v in type_props.values() if v):,}")
    return all_types, dict(type_props), dict(prop_range), dict(type_parents)


so_types, so_type_props, so_prop_range, so_type_parents = load_schemaorg_vocab()

# Sanity check: confirm all target classes are known and inheritance works.
for cls in ["ClaimReview", "Event", "NewsArticle"]:
    props = so_type_props.get(cls, set())
    print(f"  {cls}: {len(props)} properties defined (including inherited)")

## Part 4 — Filter Stores to Target-Class Entities

The raw stores contain **every subject that appeared in the subset**, not just
instances of the target class: a ClaimReview page typically also declares the
Person author, the Organization publisher, rating nodes, etc. We keep only the
entities whose `rdf:type` matches the target class (or a subclass of it).

For NewsArticle, this filtering step is doubly important — the RadioStation
store contains RadioStations, NewsArticles, ListItems, ImageObjects, sidebars,
and more. We're specifically isolating the NewsArticle subjects.

In [ ]:
def filter_by_class(store: dict, target_class: str) -> dict:
    """Keep entities whose declared type is target_class OR any descendant."""
    out = {}
    for subj, data in store.items():
        for t in data["types"]:
            local_t = t.split("/")[-1].split("#")[-1]
            # Direct type match OR target is an ancestor of the entity's type.
            if local_t == target_class or target_class in so_type_parents.get(local_t, set()):
                out[subj] = data
                break
    return out


cr_entities = filter_by_class(cr_store, "ClaimReview")
ev_entities = filter_by_class(ev_store, "Event")
na_entities = filter_by_class(rs_store, "NewsArticle")  # extracted from RadioStation dump

print(f"ClaimReview entities : {len(cr_entities):,}")
print(f"Event entities       : {len(ev_entities):,}")
print(f"NewsArticle entities : {len(na_entities):,}  (from RadioStation subset)")

# ── The DATASETS driver ──────────────────────────────────────────────────────
# Every analysis below iterates over this list. Adding a new class in the
# future is ONE line here — no per-class cells needed.
# Order matters: this is the order results will print in, top-to-bottom.
DATASETS = [
    ("ClaimReview", cr_entities),
    ("Event",       ev_entities),
    ("NewsArticle", na_entities),
]

# Seed the summary table with entity counts.
for label, ents in DATASETS:
    SUMMARY[label]["Entities"] = len(ents)

### Example entities (one from each class)

In [ ]:
def print_sample(entities: dict, label: str, n: int = 1):
    """Pretty-print the first n entities with their types, graphs, and properties."""
    print(f"\n{'═'*70}\n  {label} — {n} sample entity\n{'═'*70}")
    for i, (subj, data) in enumerate(list(entities.items())[:n]):
        print(f"[{i+1}] {subj}")
        print(f"  types  : {data['types']}")
        print(f"  graphs : {data['graphs']}")
        print(f"  props:")
        for pred, obj_graphs in data["props"].items():
            prop_name = pred.split("/")[-1]
            for obj, graphs in obj_graphs.items():
                print(f"    {prop_name:<25} = {obj[:60]}")

# One sample from each dataset — iterates DATASETS so nothing to change
# when we add a fourth class later.
for label, ents in DATASETS:
    print_sample(ents, label)

## Part 5 — Analysis Function Library

Every analysis below is written **once** as a function and applied to every
dataset in the `DATASETS` list via a simple loop. Each function also writes
its main metric into `SUMMARY` so the final comparison table is populated
automatically.

Results print in `DATASETS` order: **ClaimReview → Event → NewsArticle**.

In [ ]:
# ── Shared helper used by nearly every analysis ─────────────────────────────
def is_schema_uri(uri: str) -> bool:
    """Does this URI live in either schema.org namespace?"""
    return uri.startswith(SO_HTTP) or uri.startswith(SO_HTTPS)

def local_name(uri: str) -> str:
    """Extract the final segment of a URI (handles both / and # separators)."""
    return uri.split("/")[-1].split("#")[-1]

def get_entity_types_local(data: dict) -> set[str]:
    """All schema.org types on this entity, reduced to local names."""
    return {local_name(t) for t in data["types"] if is_schema_uri(t)}

### 5.1 Prevalence & Domain Distribution

In [ ]:
def get_pld(url: str) -> str:
    """
    Best-effort Pay-Level Domain extraction — good enough for aggregation.
    E.g. "https://www.politifact.com/factchecks/…" → "politifact.com".
    Doesn't know about complex suffixes like ".co.uk" (those show up as "co.uk"
    in output; a small caveat that doesn't affect relative rankings).
    """
    try:
        host  = urlparse(url).netloc or url
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return url


def prevalence_report(entities: dict, label: str, top_n: int = 20):
    """
    Count entities per domain. Each entity is counted once per distinct domain
    it appears on — entities that span multiple mirrors of the same publisher
    contribute +1 to that publisher, not +N.
    """
    domain_counts = Counter()
    for _, data in entities.items():
        seen = set()
        for graph in data.get("graphs", set()):
            d = get_pld(graph)
            if d not in seen:
                seen.add(d)
                domain_counts[d] += 1

    total = sum(domain_counts.values())
    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    print(f"  Total entity-domain pairs : {total:,}")
    print(f"  Unique domains            : {len(domain_counts):,}")
    print(f"\n  Top {top_n} domains by entity count:")
    df = pd.DataFrame(domain_counts.most_common(top_n), columns=["domain", "count"])
    df["% of total"] = (df["count"] / total * 100).round(2)
    print(df.to_string(index=False))

    # Record for the summary table.
    SUMMARY[label]["Unique domains"] = len(domain_counts)
    SUMMARY[label]["Top domain"]     = f"{df.iloc[0]['domain']} ({df.iloc[0]['% of total']}%)"
    return domain_counts


prevalence_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    prevalence_results[label] = prevalence_report(ents, label)
    print()  # blank line between classes

### 5.2 Namespace Correctness

Every schema.org term should be referenced with either `http://schema.org/` or
`https://schema.org/`. We flag:

- variants with typos or wrong case (`schema.Org/`, `SCHEMA.org/`)
- trailing slashes / path segments (`schema.org/Class/`)
- vocabulary terms pulled in from other ontologies (`rdf:`, `owl:`, …)

The final rate ignores non-schema namespaces (they're not *wrong* — just out of scope)
and measures: *of the URIs that are trying to be schema.org, how many actually are?*

In [ ]:
def namespace_report(entities: dict, label: str):
    """
    Split every type + property URI into three buckets and compute a correctness
    rate over (canonical + malformed). "Other" URIs (rdf:, owl:, custom vocabs)
    are reported but excluded from the denominator.
    """
    canonical = wrong = other = 0
    wrong_examples = Counter()

    for _, data in entities.items():
        # Check both types and property URIs — they live in separate maps.
        for uri in list(data["types"]) + list(data["props"].keys()):
            if is_schema_uri(uri):
                canonical += 1
            elif re.search(r'schema\.(org|org/)', uri, re.I):
                # Case-insensitive "schema.org" match that isn't the canonical form
                wrong += 1
                wrong_examples[uri] += 1
            else:
                other += 1

    total_schema = canonical + wrong
    rate = canonical / total_schema * 100 if total_schema else float("nan")

    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    print(f"  Canonical schema.org URIs : {canonical:>10,}")
    print(f"  Malformed schema.org URIs : {wrong:>10,}")
    print(f"  Other-vocab URIs          : {other:>10,}  (rdf:, owl:, etc.)")
    print(f"  Namespace correctness rate: {rate:.2f}%")
    if wrong_examples:
        print(f"  Malformed examples (top 5):")
        for uri, cnt in wrong_examples.most_common(5):
            print(f"    [{cnt:>5}]  {uri}")

    SUMMARY[label]["Namespace correctness"] = f"{rate:.2f}%"
    return rate


ns_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    ns_results[label] = namespace_report(ents, label)
    print()

### 5.3 Type Validity

For every `rdf:type` assertion that *does* use the canonical namespace, check
the local name against the full schema.org class list. Catches:

- case errors (`newsarticle` vs `NewsArticle`)
- invented types (`ExplainerNewsArticle`)
- deprecated terms that never made it into the current vocab

In [ ]:
def type_validity_report(entities: dict, label: str):
    """Check every schema-namespaced type against the vocab; report valid/invalid."""
    valid, invalid = Counter(), Counter()

    for _, data in entities.items():
        for t in data["types"]:
            if not is_schema_uri(t):
                continue  # namespace already handled in 5.2
            name = local_name(t)
            (valid if name in so_types else invalid)[name] += 1

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    print(f"  Valid types   : {sum(valid.values()):>8,}  ({rate:.2f}%)")
    print(f"  Invalid types : {sum(invalid.values()):>8,}  ({100-rate:.2f}%)")
    print(f"\n  Valid type distribution (top 10):")
    for t, c in valid.most_common(10):
        print(f"    {t:<45} {c:>8,}")
    if invalid:
        print(f"\n  Invalid types (top 10):")
        for t, c in invalid.most_common(10):
            print(f"    {t:<45} {c:>8,}")

    SUMMARY[label]["Type validity"] = f"{rate:.2f}%"
    return rate, valid, invalid


type_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    type_results[label] = type_validity_report(ents, label)  # (rate, valid, invalid)
    print()

### 5.4 Property Validity

A property use is **valid** iff the property is declared in schema.org *and*
its `domainIncludes` contains the entity's type (or any ancestor via
inheritance). We computed the transitive closure back in Part 3, so membership
here is just a set lookup.

In [ ]:
def prop_validity_report(entities: dict, label: str):
    """
    For each (entity, property) pair: is the property allowed on any of the
    entity's declared types (considering inheritance)?
    """
    valid, invalid = Counter(), Counter()

    for _, data in entities.items():
        entity_types = get_entity_types_local(data)
        # Union the "allowed" property sets across all of the entity's declared types.
        # An entity typed as both Event and SportsEvent gets the union of both.
        allowed = set()
        for et in entity_types:
            allowed |= so_type_props.get(et, set())

        for prop_uri in data["props"]:
            if not is_schema_uri(prop_uri):
                continue
            name = local_name(prop_uri)
            (valid if name in allowed else invalid)[name] += 1

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    print(f"  Valid property uses   : {sum(valid.values()):>10,}  ({rate:.2f}%)")
    print(f"  Invalid property uses : {sum(invalid.values()):>10,}  ({100-rate:.2f}%)")
    print(f"\n  Most used valid properties (top 15):")
    for p, c in valid.most_common(15):
        print(f"    {p:<45} {c:>10,}")
    if invalid:
        print(f"\n  Invalid properties (top 10):")
        for p, c in invalid.most_common(10):
            print(f"    {p:<45} {c:>10,}")

    SUMMARY[label]["Property validity"] = f"{rate:.2f}%"
    return rate, valid, invalid


prop_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    prop_results[label] = prop_validity_report(ents, label)  # (rate, valid, invalid)
    print()

### 5.5 Domain/Range Violations

For each value, check whether it matches the property's expected range:

| Property kind       | Expected shape                                  |
|---------------------|-------------------------------------------------|
| Date/time           | ISO-8601 literal (`2024-01-15`, `…T12:00:00Z`)  |
| URL-typed           | Actual URI (`<http…>`) or `http…` literal        |
| Object-valued       | URI or blank node (**not** a string literal)     |
| Literal-only        | Always valid (not checked here)                  |

Only properties the vocab tells us about are checked; others are ignored.

In [ ]:
# Properties that must carry dates/times — sampled from the most common in practice.
DATE_PROPS = {"datePublished", "dateModified", "dateCreated", "expires",
              "startDate", "endDate", "uploadDate"}

# Properties whose value must dereference (or at least look like) a URL.
URL_PROPS  = {"url", "sameAs", "mainEntityOfPage", "image", "thumbnail"}

# Schema.org datatypes — if rangeIncludes is entirely within this set,
# a literal object is fine and we don't flag it.
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean",
                     "Date", "DateTime", "Time", "URL", "Duration", "Mass",
                     "Energy", "Distance", "Speed", "XPathType", "CssSelectorType"}

# Permissive ISO-8601 matcher: covers year-only, date-only, and full datetime
# with optional seconds, microseconds, and timezone suffix.
ISO_DATE_RE = re.compile(
    r'^\d{4}(-\d{2}(-\d{2}(T\d{2}:\d{2}(:\d{2}(\.\d+)?)?([+-]\d{2}:\d{2}|Z)?)?)?)?$'
)

def is_literal(obj: str) -> bool:
    return obj.startswith('"')

def strip_literal(obj: str) -> str:
    """Extract the text content of a quoted literal; no-op for non-literals."""
    m = re.match(r'^"((?:[^"\\]|\\.)*)"', obj)
    return m.group(1) if m else obj


def range_violation_report(entities: dict, label: str):
    """
    Classify every value against its property's expected range.
    Returns violation_rate and a Counter of offending properties.
    """
    violations = Counter()
    ok         = Counter()
    viol_examples = defaultdict(list)

    for _, data in entities.items():
        for prop_uri, values in data["props"].items():
            if not is_schema_uri(prop_uri):
                continue
            prop_local = local_name(prop_uri)
            ranges     = so_prop_range.get(prop_local, set())

            # "Entity-expected" = at least one range element is a class, not a datatype.
            expects_entity = bool(ranges - LITERAL_DATATYPES) and not ranges.issubset(LITERAL_DATATYPES)
            is_date_prop   = prop_local in DATE_PROPS
            is_url_prop    = prop_local in URL_PROPS

            for obj in values:
                if is_date_prop:
                    val = strip_literal(obj)
                    if ISO_DATE_RE.match(val):
                        ok[prop_local] += 1
                    else:
                        violations[prop_local] += 1
                        if len(viol_examples[prop_local]) < 3:
                            viol_examples[prop_local].append(f"date={val!r}")
                elif is_url_prop:
                    # A URL is fine as a URI, a blank node, OR a literal that starts with http
                    if is_literal(obj) and not strip_literal(obj).startswith("http"):
                        violations[prop_local] += 1
                        if len(viol_examples[prop_local]) < 3:
                            viol_examples[prop_local].append(f"url={strip_literal(obj)[:60]!r}")
                    else:
                        ok[prop_local] += 1
                elif expects_entity:
                    # Object-valued property with a literal = flagged
                    if is_literal(obj):
                        violations[prop_local] += 1
                        if len(viol_examples[prop_local]) < 3:
                            viol_examples[prop_local].append(f"lit={strip_literal(obj)[:60]!r}")
                    else:
                        ok[prop_local] += 1
                # else: pure-literal property → no check applicable, skip.

    total = sum(violations.values()) + sum(ok.values())
    rate  = sum(violations.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    print(f"  Checked value assertions : {total:>10,}")
    print(f"  Range violations         : {sum(violations.values()):>10,}  ({rate:.2f}%)")
    print(f"\n  Top violation properties (top 10):")
    for p, c in violations.most_common(10):
        exs = viol_examples.get(p, [])
        print(f"    {p:<35} {c:>8,}  e.g. {exs}")

    SUMMARY[label]["Range violation rate"] = f"{rate:.2f}%"
    return rate, violations


range_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    range_results[label] = range_violation_report(ents, label)  # (rate, violations)
    print()

### 5.6 Property Coverage

Of all properties the schema.org vocabulary *defines* for this class, how many
are actually used by at least one real entity? Low coverage means publishers
are ignoring most of the vocabulary the spec provides — common in practice.

In [ ]:
def coverage_report(entities: dict, target_class: str):
    """Fraction of class-valid properties that are used at least once in the wild."""
    defined_props = so_type_props.get(target_class, set())
    if not defined_props:
        print(f"  No properties defined in vocab for {target_class}")
        return None, None

    # Properties actually observed on any entity
    used_props = set()
    for data in entities.values():
        for prop in data["props"]:
            if is_schema_uri(prop):
                used_props.add(local_name(prop))

    used_and_defined  = used_props & defined_props
    defined_not_used  = defined_props - used_props
    coverage_pct      = len(used_and_defined) / len(defined_props) * 100

    print(f"\n{'─'*60}\n  {target_class} — Property Coverage\n{'─'*60}")
    print(f"  Properties defined in schema.org : {len(defined_props):>5}")
    print(f"  Properties used (≥ 1 entity)     : {len(used_and_defined):>5}  ({coverage_pct:.1f}% coverage)")
    print(f"  Properties never used            : {len(defined_not_used):>5}  ({100-coverage_pct:.1f}% gap)")

    # Density = what fraction of entities use this property.
    # Answers: "is it a universally-used core property, or a long-tail one?"
    prop_freq = Counter()
    for data in entities.values():
        for prop in data["props"]:
            if is_schema_uri(prop):
                p = local_name(prop)
                if p in defined_props:
                    prop_freq[p] += 1
    n = len(entities) or 1

    rows = [(p, prop_freq[p], prop_freq[p]/n*100) for p in defined_props]
    rows.sort(key=lambda x: -x[1])
    df = pd.DataFrame(rows, columns=["property", "entities_using", "density_%"])
    df["density_%"] = df["density_%"].round(2)

    print(f"\n  Used defined properties (by density):")
    print(df[df["entities_using"] > 0].to_string(index=False))
    if defined_not_used:
        # Truncate the never-used list for Event (could be 700+ items)
        nu = sorted(defined_not_used)
        preview = ", ".join(nu[:20]) + (f", … ({len(nu)-20} more)" if len(nu) > 20 else "")
        print(f"\n  Never-used properties ({len(nu)}): {preview}")

    SUMMARY[target_class]["Property coverage"] = f"{coverage_pct:.1f}%"
    return coverage_pct, df


coverage_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    coverage_results[label] = coverage_report(ents, label)  # (pct, df)
    print()

### 5.7 AvP — Average Properties per Entity

AvP counts distinct schema.org properties per entity (multi-valued properties
count once). A low AvP relative to vocabulary size is common — most publishers
fill in a small core set and nothing else.

In [ ]:
def avp_report(entities: dict, target_class: str):
    """
    Compute per-entity property counts and print a summary + histogram.
    (Histogram bug from the previous version — printing a duplicate "20+: 0" row —
     is fixed here by clamping to 20 once and iterating over 0..20 exactly.)
    """
    defined_props = so_type_props.get(target_class, set())
    n_defined     = len(defined_props) or 1

    prop_counts = []
    for data in entities.values():
        so_props_used = {
            local_name(prop) for prop in data["props"] if is_schema_uri(prop)
        }
        prop_counts.append(len(so_props_used))

    if not prop_counts:
        print(f"  No entities for {target_class}")
        return None

    avp = statistics.mean(prop_counts)
    med = statistics.median(prop_counts)
    std = statistics.stdev(prop_counts) if len(prop_counts) > 1 else 0.0
    mn, mx = min(prop_counts), max(prop_counts)

    print(f"\n{'─'*60}\n  {target_class} — Average Properties per Entity (AvP)\n{'─'*60}")
    print(f"  Entities          : {len(prop_counts):>8,}")
    print(f"  Available props   : {n_defined:>8,}  (schema.org defined)")
    print(f"  AvP (mean)        : {avp:>8.2f}  ({avp/n_defined*100:.1f}% of available)")
    print(f"  Median            : {med:>8.1f}")
    print(f"  Std dev           : {std:>8.2f}")
    print(f"  Min / Max         : {mn:>4} / {mx:>4}")

    # Histogram: bucket everything ≥20 into a single "20+" bucket.
    buckets = Counter()
    for c in prop_counts:
        buckets[c if c < 20 else 20] += 1

    max_bar = max(buckets.values(), default=1)
    print(f"\n  Distribution (# distinct schema.org properties per entity):")
    # Iterate 0..20 exactly — no more, no less. This fixes the duplicate "20+"
    # that the original loop emitted when mx < 20.
    for k in range(0, 21):
        bar_len = buckets.get(k, 0) * 40 // max_bar
        label   = f"{k:>2}" if k < 20 else "20+"
        print(f"    {label}: {'█' * bar_len} {buckets.get(k, 0):,}")

    SUMMARY[target_class]["AvP (mean)"] = f"{avp:.2f}"
    SUMMARY[target_class]["AvP (median)"] = f"{med:.0f}"
    return {"mean": avp, "median": med, "std": std, "min": mn, "max": mx}


avp_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    avp_results[label] = avp_report(ents, label)
    print()

### 5.8 Characteristic Sets

A *characteristic set* (Neumann & Moerkotte 2011) is the exact set of
properties an entity has populated. The most frequent sets reveal the *actual*
annotation patterns publishers follow, independent of what schema.org allows.

If only 5 sets cover 80% of entities, the corpus is highly templated
(typically a few big publishers with identical CMS output). A long tail means
diverse, hand-curated annotations.

In [ ]:
def characteristic_sets_report(entities: dict, label: str, top_n: int = 15):
    """
    Group entities by the exact frozenset of schema.org property local names
    they use, and report the most frequent groups.
    """
    set_counter = Counter()

    for data in entities.values():
        so_props = frozenset(
            local_name(prop) for prop in data["props"] if is_schema_uri(prop)
        )
        set_counter[so_props] += 1

    total = len(entities) or 1

    print(f"\n{'─'*60}\n  {label} — Characteristic Sets (top {top_n})\n{'─'*60}")
    print(f"  Unique property sets : {len(set_counter):,}")
    # Concentration metric: what fraction of entities fall into just the top_n sets?
    top_coverage = sum(c for _, c in set_counter.most_common(top_n)) / total * 100
    print(f"  Top {top_n} cover      : {top_coverage:.1f}% of entities")
    print()
    for i, (prop_set, count) in enumerate(set_counter.most_common(top_n), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no SO properties)"
        print(f"  #{i:>2}  [{count:>7,}  {pct:>5.1f}%]  {{{props_str}}}")

    SUMMARY[label]["Unique char. sets"] = len(set_counter)
    SUMMARY[label][f"Top {top_n} sets cover"] = f"{top_coverage:.1f}%"
    return set_counter


char_sets_results = {}
for label, ents in DATASETS:
    print(f"══ {label} ══")
    char_sets_results[label] = characteristic_sets_report(ents, label)
    print()

## Part 6 — Consolidated Summary

Every analysis above wrote its headline numbers into the global `SUMMARY` dict.
Here we render them as a single side-by-side table across all classes in `DATASETS`.

In [ ]:
# Pick a stable row order so the table reads top-to-bottom like a scorecard.
ROW_ORDER = [
    "Entities",
    "Unique domains",
    "Top domain",
    "Namespace correctness",
    "Type validity",
    "Property validity",
    "Range violation rate",
    "Property coverage",
    "AvP (mean)",
    "AvP (median)",
    "Unique char. sets",
    "Top 15 sets cover",
]

# Build the table by iterating DATASETS so the column count adapts automatically.
rows = []
for metric in ROW_ORDER:
    row = {"Metric": metric}
    for label, _ in DATASETS:
        row[label] = SUMMARY[label].get(metric, "—")
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("Side-by-side comparison\n" + "=" * 80)
print(summary_df.to_string(index=False))

### Interpretation cheat sheet

- **High namespace correctness + high type validity** → publishers know the vocab.
- **Low property validity** → terms being applied to the wrong class (domain mismatch).
- **High range violations** → publishers writing strings where objects are expected
  (common for `datePublished`, `image`, `author`).
- **Low property coverage with high AvP** → the few properties used are used heavily
  (templated output); small core of the vocab dominates.
- **Few characteristic sets covering >50% of entities** → dominated by a handful
  of CMS templates (e.g. Google-recommended factcheck layouts).

### ⚠️ A note on the NewsArticle numbers

The NewsArticle column reflects articles **only on pages that also host
RadioStation markup** — i.e. published on radio-station websites. This is a
narrow slice of NewsArticle on the web at large. Expect it to look more uniform
than a broad NewsArticle sample would:

- Property coverage may be artificially low (radio CMSs use a restrictive
  template set).
- Characteristic sets may concentrate heavily in the top few (one or two
  CMS vendors likely dominate the radio-site market).
- Top domains will be radio-industry hosts, not mainstream news publishers.

For a representative NewsArticle analysis you'd need either a dedicated
NewsArticle subset (which WDC does not publish), a re-extraction from Common
Crawl, or filtering NewsArticle out of other related-class subsets (e.g.
Organization, Person, CreativeWork) and unioning them — noting the same
bias caveat applies to each.